In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from retrieve import load_translation_memory, fuzzy_retrieval

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [8]:
# Test case
translation_memory_path = "../data/tm/translation_memory.jsonl"
new_source = "The required traits for most monsters are as follows:"
tm_match = fuzzy_retrieval(translation_memory, new_source, top_n=1)
print(tm_match)

[(81.55339805825243, {'id': '0974de15d436510c973511c3157674a2', 'source_file': 'pl_manual.po', 'source': 'The possible traits for most units are as follows:', 'target': 'Większość jednostek może posiadać następujące cechy:', 'msgctxt': None, 'wml_context': 'type: Content of: <book><chapter><section><section><simpara>', 'occurrences': [['doc/manual/manual.en.xml', '888']], 'flags': []})]


In [16]:
source = tm_match[0][1]["source"]
target_language = "Polish"
print(f"Source: {source}")

Source: The possible traits for most units are as follows:


In [44]:
def build_repair_prompt(new_source: str, target_language: str, translation_memory_path: str):
    translation_memory = load_translation_memory(translation_memory_path)
    tm_match = fuzzy_retrieval(translation_memory, new_source, top_n=1)
    tm_source = tm_match[0][1]["source"]
    tm_target = tm_match[0][1]["target"]

    PROMPT = f"""You are a professional video games translator that translates text from English to {target_language}. 
Use the pair ({tm_source}, {tm_target}) as a reference for translating {new_source} into {target_language}.
Replace with {target_language} translation only those words that are different from the reference. Do not translate words that are the same as the reference.
You must maintain the same formatting, tags, and placeholders as in the source text. Do not add any additional commentary or explanation."""
    return PROMPT

In [30]:
print(build_repair_prompt(new_source, target_language, translation_memory_path))

You are a professional video games translator that translates text from English to Polish. 
Use the pair (The possible traits for most units are as follows:, Większość jednostek może posiadać następujące cechy:) as a reference for translating The required traits for most monsters are as follows: into Polish.
Replace with Polish translation only those words that are different from the reference. Do not translate words that are the same as the reference.
You must maintain the same formatting, tags, and placeholders as in the source text. Do not add any additional commentary or explanation.


In [34]:
def call_repair(new_source: str, target_language: str, translation_memory_path: str):
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    role = "system"
    content = build_repair_prompt(new_source, target_language, translation_memory_path)

    model = "gpt-4o"
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": role, "content": content}],
        temperature=0
    )
    return response.choices[0].message.content.strip()



In [35]:
print(call_repair(new_source, target_language, translation_memory_path))

Wymagane cechy dla większości potworów są następujące:


## Testing the model with new strings

In [37]:
test1 = "Elvish Runemaster"
test2 = "Strong units do 1 more damage for every successful strike in melee combat.  Strength is a trait possessed only by Humans. The Humans are known for their reso;oamce, and their great facility with swords. Some, however, are gifted with natural talent that exceeds their brethren. These humans inflict an additional point of damage with sword strike."
test3 = "Young lady, you have $student_hp hitpoints and a javelin. I’m fairly sure you’ll succeed."

In [45]:
print(build_repair_prompt(test1, target_language, translation_memory_path))
print(call_repair(test1, target_language, translation_memory_path))

You are a professional video games translator that translates text from English to Polish. 
Use the pair (Dwarvish Runemaster, Krasnoludzki runmistrz) as a reference for translating Elvish Runemaster into Polish.
Replace with Polish translation only those words that are different from the reference. Do not translate words that are the same as the reference.
You must maintain the same formatting, tags, and placeholders as in the source text. Do not add any additional commentary or explanation.
Elfi runmistrz


In [46]:
print(build_repair_prompt(test2, target_language, translation_memory_path))
print(call_repair(test2, target_language, translation_memory_path))

You are a professional video games translator that translates text from English to Polish. 
Use the pair (Dextrous units do 1 more damage for every successful strike in ranged combat.  Dextrous is a trait possessed only by Elves. The Elven people are known for their uncanny grace, and their great facility with the bow. Some, however, are gifted with natural talent that exceeds their brethren. These elves inflict an additional point of damage with each arrow., Jednostki zwinne zadają o 1 punkt obrażeń więcej każdym trafieniem w walce na odległość. Zwinność jest cechą dostępną wyłącznie dla elfów, które słynną z niezwykłej gracji poruszania sie i umiejętności doskonałego posługiwania się łukiem. Niektóre elfy mają jednak naturalne zdolności przekraczające nawet umiejętności ich pobratymców - każda ich strzała zadaje o jeden punkt obrażeń więcej.) as a reference for translating Strong units do 1 more damage for every successful strike in melee combat.  Strength is a trait possessed only b

In [47]:
print(build_repair_prompt(test3, target_language, translation_memory_path))
print(call_repair(test3, target_language, translation_memory_path))

You are a professional video games translator that translates text from English to Polish. 
Use the pair (Young man, you have $student_hp hitpoints and a sword. I’m fairly sure you’ll win., Młodzieńcze, masz $student_hp punkty życia i miecz. Jestem pewien, że zwyciężysz.) as a reference for translating Young lady, you have $student_hp hitpoints and a javelin. I’m fairly sure you’ll succeed. into Polish.
Replace with Polish translation only those words that are different from the reference. Do not translate words that are the same as the reference.
You must maintain the same formatting, tags, and placeholders as in the source text. Do not add any additional commentary or explanation.
Young lady, you have $student_hp hitpoints and a oszczep. I’m fairly sure you’ll succeed.
